# Prerequisite 02 · NumPy：shape 是第一等公民

[上一章：Python](Prerequisite01_Python.ipynb) · [下一章：Matplotlib](Prerequisite03_Matplotlib.ipynb)

**目标：**熟练处理 state、batch 和 action 维度，理解广播与高级索引。RL 中大量“算法错误”其实是 shape 悄悄错了。


## 1. 数组、shape、dtype

一个状态可能是 `(features,)`，一批状态通常是 `(batch, features)`，Q 网络输出通常是 `(batch, actions)`。


In [1]:
import numpy as np

states = np.array([[0.1, 0.2, -0.3], [0.0, 0.4, 0.5]], dtype=np.float32)
weights = np.array([[1.0, -1.0], [0.5, 0.2], [-0.3, 0.8]], dtype=np.float32)
q_values = states @ weights
print("states:", states.shape, states.dtype)
print("Q(s,a):", q_values.shape, "\n", q_values)
assert q_values.shape == (2, 2)


states: (2, 3) float32
Q(s,a): (2, 2) 
 [[ 0.29000002 -0.3       ]
 [ 0.05        0.48000002]]


## 2. axis 与动作索引

`axis=1` 表示在每个样本的动作维上归约。训练 DQN 时，需要从每行 Q 值中取出实际执行动作对应的一项。


In [2]:
actions = np.array([1, 0])
batch_index = np.arange(len(actions))
chosen_q = q_values[batch_index, actions]
greedy_actions = q_values.argmax(axis=1)

print("chosen Q:", chosen_q)
print("greedy actions:", greedy_actions)
print("max next Q:", q_values.max(axis=1))


chosen Q: [-0.3   0.05]
greedy actions: [0 1]
max next Q: [0.29000002 0.48000002]


## 3. 广播：方便，也危险

`(B,A)` 加 `(B,1)` 会把每个样本的标量复制到所有动作；加 `(B,)` 通常无法表达你想要的语义。先在纸上写 shape，再写代码。


In [3]:
advantages = np.array([[1.0, 2.0], [-1.0, 3.0]])
values = np.array([[10.0], [20.0]])
dueling_q = values + advantages - advantages.mean(axis=1, keepdims=True)
print(dueling_q)
assert np.allclose(dueling_q.mean(axis=1), values[:, 0])


[[ 9.5 10.5]
 [18.  22. ]]


## 4. 随机数与可复现

推荐使用 `np.random.default_rng(seed)`，并把生成器显式传入实验边界。相同 seed 是调试工具，不代表一次运行就足以得出结论。


In [4]:
rng_a = np.random.default_rng(42)
rng_b = np.random.default_rng(42)
sample_a = rng_a.normal(size=4)
sample_b = rng_b.normal(size=4)
print(sample_a)
assert np.allclose(sample_a, sample_b)


[ 0.30471708 -1.03998411  0.7504512   0.94056472]


## 自动批改练习

每道题先读题、再写代码、最后运行同一格。看到 `[通过]` 才算完成；`[未通过]` 会说明哪里不对。
参考答案默认折叠，建议独立尝试至少 10 分钟后再看。


In [5]:
# 自动检查工具：每道练习末尾都会调用 check(test_xxx)。
# 未完成或有错时只打印提示，不会中断 Run All。
def check(test):
    try:
        test()
    except NotImplementedError:
        print(f"[未完成] {test.__name__}：把 raise NotImplementedError 换成你的代码，再运行本格。")
    except AssertionError as error:
        print(f"[未通过] {test.__name__}：{error}")
    except Exception as error:  # 代码本身报错，也给出可读提示
        print(f"[代码报错] {test.__name__}：{type(error).__name__}: {error}")
    else:
        print(f"[通过] {test.__name__}")


### 练习 1：批量取出已选动作的 Q 值

实现 `select_actions(q, actions)`：`q` 为 `(B,A)`，`actions` 为 `(B,)` 整数，返回 `(B,)`，第 i 项是 `q[i, actions[i]]`。**不要用 for 循环。**

<details><summary>提示（卡住 10 分钟再看）</summary>

`q[np.arange(len(actions)), actions]`。

</details>


In [6]:
import numpy as np

def select_actions(q, actions):
    raise NotImplementedError


# ---- 自动检查（不用修改） ----
import numpy as np

def test_select_actions():
    q = np.array([[1., 4., 2.], [5., 0., 3.], [2., 1., 6.], [0., 9., 8.]])
    out = select_actions(q, np.array([2, 0, 1, 1]))
    assert out.shape == (4,), f"输出应为 (4,)，得到 {out.shape}"
    assert np.allclose(out, [2., 5., 1., 9.]), f"应为 [2,5,1,9]，得到 {out}"

check(test_select_actions)


[未完成] test_select_actions：把 raise NotImplementedError 换成你的代码，再运行本格。


<details><summary>参考答案（先独立完成，再对照）</summary>

```python
import numpy as np

def select_actions(q, actions):
    return q[np.arange(len(actions)), actions]
```

</details>


### 练习 2：one-hot 编码

实现 `one_hot(indices, n)`：输入 `(B,)` 整数数组，返回 `(B,n)` 的 float 数组，每行只有对应位置为 1。

<details><summary>提示（卡住 10 分钟再看）</summary>

先 `out = np.zeros((len(indices), n))`，再用上一题的高级索引赋值为 1。

</details>


In [7]:
import numpy as np

def one_hot(indices, n):
    raise NotImplementedError


# ---- 自动检查（不用修改） ----
import numpy as np

def test_one_hot():
    out = one_hot([2, 0], 3)
    assert out.shape == (2, 3), f"应为 (2,3)，得到 {out.shape}"
    assert np.array_equal(out, [[0, 0, 1], [1, 0, 0]]), f"结果不对：{out}"

check(test_one_hot)


[未完成] test_one_hot：把 raise NotImplementedError 换成你的代码，再运行本格。


<details><summary>参考答案（先独立完成，再对照）</summary>

```python
import numpy as np

def one_hot(indices, n):
    indices = np.asarray(indices)
    out = np.zeros((len(indices), n))
    out[np.arange(len(indices)), indices] = 1.0
    return out
```

</details>


### 练习 3：移动平均

实现 `moving_average(x, window)`：返回长度 `len(x)-window+1` 的数组，第 i 项是 `x[i:i+window]` 的平均。训练曲线几乎都要先平滑再看。

<details><summary>提示（卡住 10 分钟再看）</summary>

`np.convolve(x, np.ones(window) / window, mode="valid")`，或者用 `np.cumsum`。

</details>


In [8]:
import numpy as np

def moving_average(x, window):
    raise NotImplementedError


# ---- 自动检查（不用修改） ----
import numpy as np

def test_moving_average():
    out = moving_average([1, 2, 3, 4, 5], 2)
    assert np.allclose(out, [1.5, 2.5, 3.5, 4.5]), f"应为 [1.5,2.5,3.5,4.5]，得到 {out}"
    assert len(moving_average(np.arange(100), 10)) == 91, "长度应为 len(x)-window+1"

check(test_moving_average)


[未完成] test_moving_average：把 raise NotImplementedError 换成你的代码，再运行本格。


<details><summary>参考答案（先独立完成，再对照）</summary>

```python
import numpy as np

def moving_average(x, window):
    x = np.asarray(x, dtype=float)
    return np.convolve(x, np.ones(window) / window, mode="valid")
```

</details>


### 练习 4：只用训练集统计量做标准化

实现 `fit_normalizer(train)`，返回函数 `normalize(x)`：按**列**（每个特征）减去训练集均值、除以训练集标准差（加 `1e-8` 防止除 0）。
测试集必须用训练集的统计量，否则就是数据泄漏。


In [9]:
import numpy as np

def fit_normalizer(train):
    raise NotImplementedError


# ---- 自动检查（不用修改） ----
import numpy as np

def test_fit_normalizer():
    rng = np.random.default_rng(0)
    train = rng.normal([5.0, -3.0], [2.0, 0.5], size=(1000, 2))
    norm = fit_normalizer(train)
    z = norm(train)
    assert np.allclose(z.mean(axis=0), 0, atol=1e-6) and np.allclose(z.std(axis=0), 1, atol=1e-3), "训练集标准化后每列应均值 0、标准差 1"
    test = np.array([[5.0, -3.0]])
    assert np.allclose(norm(test), 0, atol=0.1), "测试数据应使用训练集的均值和标准差"

check(test_fit_normalizer)


[未完成] test_fit_normalizer：把 raise NotImplementedError 换成你的代码，再运行本格。


<details><summary>参考答案（先独立完成，再对照）</summary>

```python
import numpy as np

def fit_normalizer(train):
    mean = train.mean(axis=0)
    std = train.std(axis=0) + 1e-8
    def normalize(x):
        return (x - mean) / std
    return normalize
```

</details>


## 练习与过关标准

**练习：**给定 `q.shape == (4,3)` 和 `actions == [2,0,1,2]`，不用循环取出四个已选动作的 Q 值。

<details><summary>答案</summary>`q[np.arange(4), actions]`。结果 shape 应为 `(4,)`。</details>

**过关：**看到数组运算能说出输入、输出 shape；会使用 `reshape`、布尔 mask、`axis`、矩阵乘法和 action indexing。

**官方选读：**[NumPy absolute basics](https://numpy.org/doc/stable/user/absolute_beginners)；需要更系统时读 [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html)。
